In [153]:
from file_reader import CSVReader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score
from sklearn import svm
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from collections import Counter
import re
from matplotlib import pyplot as plt
import seaborn as sns # for nice visualizations


reader = CSVReader()
recordings = reader.get_data_files_as_df()


def convert_gyro_data():
    need_gyro_conversion = ["lennart", "maximilian"]
    for recording in recordings:
        if recording["name"] in need_gyro_conversion:
            recording["data"]["gyro_x"] = np.deg2rad(recording["data"]["gyro_x"])
            recording["data"]["gyro_y"] = np.deg2rad(recording["data"]["gyro_y"])
            recording["data"]["gyro_z"] = np.deg2rad(recording["data"]["gyro_z"])


def split_recordings_to_train_test(whole_set):
    training, test = train_test_split(
        whole_set,
        test_size=0.2,
        stratify=[
            f"{r['activity']}_{r['name']}_{r['sensor_placement']}_{r['sample_rate']}"
            for r in recordings
        ],
        random_state=42,
    )
    return training, test


def create_time_windows(recordings, window_length):
    windows = []

    for rec in recordings:
        data = rec["data"]
        activity = rec["activity"]
        sample_rate = int(rec["sample_rate"].replace("hz", ""))
        window_size = sample_rate * window_length

        window_start = 0
        window_end = window_size

        while window_end <= len(data):
            window = data.iloc[window_start:window_end].copy()
            window["activity"] = activity
            window["sample"] = sample_rate
            windows.append(window)

            window_start += window_size
            window_end += window_size

    return windows


def transform_windows_to_features(windows):
    feature_rows = []
    
    for window in windows:
        feature_row = {}
        
        feature_row["activity"] = window["activity"].iloc[0]
        
        for acc_col in ["acc_x", "acc_y", "acc_z"]:
            feature_row[f"{acc_col}_mean"] = window[acc_col].mean()
            feature_row[f"{acc_col}_std"] = window[acc_col].std()
            feature_row[f"{acc_col}_min"] = window[acc_col].min()
            feature_row[f"{acc_col}_max"] = window[acc_col].max()

        for gyro_col in ["gyro_x", "gyro_y", "gyro_z"]:
            feature_row[f"{gyro_col}_mean"] = window[gyro_col].mean()
            feature_row[f"{gyro_col}_std"] = window[gyro_col].std()
            feature_row[f"{gyro_col}_min"] = window[gyro_col].min()
            feature_row[f"{gyro_col}_max"] = window[gyro_col].max()

        acc_strengths = np.sqrt(window["acc_x"]**2 + window["acc_y"]**2 + window["acc_z"]**2)
        gyro_strengths = np.sqrt(window["gyro_x"]**2 + window["gyro_y"]**2 + window["gyro_z"]**2)
        
        feature_row["acc_strenght_mean"] = acc_strengths.mean()
        feature_row["acc_strenght_std"] = acc_strengths.std()
        
        feature_row["gyro_strenght_mean"] = gyro_strengths.mean()
        feature_row["gyro_strenght_std"] = gyro_strengths.std()
        
        signal = acc_strengths - acc_strengths.mean()
        signal_hamming = signal * np.hamming(len(signal))
        acc_fft = np.fft.rfft(signal_hamming)
        acc_freqs = np.fft.rfftfreq(len(signal_hamming), 1/window["sample"].iloc[0])
        #magnitudes = np.abs(acc_fft)
        #dominant_freq_index = np.argmax(magnitudes[1:]) + 1  # Skip the zero frequency component
        #feature_row["acc_dom_freq"] = acc_freqs[dominant_freq_index]
        feature_row["acc_dom_freq"] = acc_freqs[np.argmax(np.abs(acc_fft))]
        
        
        signal = gyro_strengths - gyro_strengths.mean()
        signal_hamming = signal * np.hamming(len(signal))
        gyro_fft = np.fft.rfft(signal_hamming)
        gyro_freqs = np.fft.rfftfreq(len(signal_hamming), 1/window["sample"].iloc[0]) 
        feature_row["dom_gyro_freq"] = gyro_freqs[np.argmax(np.abs(gyro_fft))]
        
        feature_rows.append(feature_row)
    
    classifier_data = pd.DataFrame(feature_rows)
    return classifier_data


def perform_standard_scaling(train_df, df_to_scale):
    scaler = StandardScaler()
    scaler.fit(train_df[[col for col in train_df.columns if col != "activity"]])
    scaled_samples =  scaler.transform(df_to_scale[[col for col in df_to_scale.columns if col != "activity"]])
    df_scaled = df_to_scale.copy()
    df_scaled[[col for col in df_scaled.columns if col != "activity"]] = scaled_samples
    return df_scaled

def perform_normalization(train_df, df_to_normalize):
    scaler = MinMaxScaler()
    scaler.fit(train_df[[col for col in train_df.columns if col != "activity"]])
    scaled_samples = scaler.transform(df_to_normalize[[col for col in df_to_normalize.columns if col != "activity"]])
    df_normalized = df_to_normalize.copy()
    df_normalized[[col for col in df_normalized.columns if col != "activity"]] = scaled_samples
    return df_normalized

convert_gyro_data()

train_recordings, test_recording = split_recordings_to_train_test(recordings)
# split before standardi

# classifier mit normalized_train und normalized_test trainieren usw


In [155]:
def train_and_evaluate(classifier, features_train, classes_train, features_test, classes_test):
    classifier.fit(features_train, classes_train)

    # Predict test data
    classes_predicted = classifier.predict(features_test)

    # Calculate accuracy
    accuracy = accuracy_score(classes_test, classes_predicted)
    
    # Calculate macro averaged F1 score
    f1 = f1_score(classes_test, classes_predicted, average='macro')
    return classes_predicted, accuracy, f1

In [156]:
def plot_confusion_matrix(classes_test, classes_predicted, ax=None):
    conf_matrix = confusion_matrix(classes_test, classes_predicted)

    ConfusionMatrixDisplay(conf_matrix, display_labels=np.unique(classes_test)).plot(ax=ax)

    plt.xticks(rotation=90, ha='center')

In [157]:
def plot_classification_report(classes_test, classes_predicted):
    report = classification_report(classes_test, classes_predicted, output_dict=True)
    df = pd.DataFrame(report)
    return df

In [158]:
def evaluate_classifiers(x_train, y_train, x_test, y_test, print_confusion_matrices=False, print_classification_reports=False):
    strategies = {
        "normal": lambda kernel: svm.SVC(kernel=kernel),
        "one_vs_one": lambda kernel: OneVsOneClassifier(svm.SVC(kernel=kernel)),
        "one_vs_rest": lambda kernel: OneVsRestClassifier(svm.SVC(kernel=kernel)),
    }

    classifiers = []
    predictions_list = []

    for strategy_name, function in strategies.items():
        for selected_kernel in ["linear", "poly", "rbf", "sigmoid"]:
            classifier = function(selected_kernel)
            classifiers.append(classifier)
            predictions, accuracy, f1 = train_and_evaluate(
                classifier, x_train, y_train, x_test, y_test)
            predictions_list.append(predictions)
            print(
                f"{strategy_name} | {selected_kernel.capitalize()} | Accuracy: {accuracy:.4f} | F1 Macro: {f1:.4f}")

            report_df = plot_classification_report(y_test, predictions).drop(
                columns=["macro avg", "weighted avg", "accuracy"]).round(3).transpose()

        if print_confusion_matrices:
            fig, axes = plt.subplots(1, 4, figsize=(22, 5))

            for i, selected_kernel in enumerate(["linear", "poly", "rbf", "sigmoid"]):
                plot_confusion_matrix(y_test, predictions_list[i], ax=axes[i])
                axes[i].set_title(
                    f"{strategy_name} | {selected_kernel.capitalize()}")
            plt.tight_layout()

        

        if print_classification_reports:
            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            for i, selected_kernel in enumerate(["linear", "poly", "rbf", "sigmoid"]):
                report_df = plot_classification_report(y_test, predictions_list[i]).drop(
                    columns=["macro avg", "weighted avg", "accuracy"]).round(2).transpose()
                report_df.columns = ["prec", "rec", "f1", "sup"]
                axes[i].axis("off")
                table = axes[i].table(
                    cellText=report_df.values,
                    colLabels=report_df.columns,
                    rowLabels=report_df.index,
                    loc='center'
                )
                axes[i].set_title(
                    f"{strategy_name} | {selected_kernel.capitalize()}")
                table.auto_set_font_size(False)
                table.set_fontsize(14)
                table.scale(1.1, 3)

            plt.tight_layout()

## Complete Data Set ##

### 1. Test (Complete) ###
- Window Length 2
- Complete Features 
- Sample Rate 20Hz and 100Hz
- Sensor Placement Hand and Pocket

In [159]:
WINDOW_LENGTH = 2

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

classifier_train_data = transform_windows_to_features(train_windows)
classifier_test_data = transform_windows_to_features(test_windows)

standard_scaled_train = perform_standard_scaling(classifier_train_data, classifier_train_data)
normalized_train = perform_normalization(standard_scaled_train, standard_scaled_train )

standard_scaled_test = perform_standard_scaling(classifier_train_data, classifier_test_data)
normalized_test = perform_normalization(standard_scaled_train, standard_scaled_test)


In [160]:
x_train = standard_scaled_train.drop(columns=["activity"])
y_train = standard_scaled_train["activity"]

x_test = standard_scaled_test.drop(columns=["activity"])
y_test = standard_scaled_test["activity"]

In [161]:
evaluate_classifiers(x_train, y_train, x_test, y_test)

normal | Linear | Accuracy: 0.8374 | F1 Macro: 0.8360
normal | Poly | Accuracy: 0.7239 | F1 Macro: 0.7173
normal | Rbf | Accuracy: 0.8813 | F1 Macro: 0.8776
normal | Sigmoid | Accuracy: 0.5535 | F1 Macro: 0.5541
one_vs_one | Linear | Accuracy: 0.8426 | F1 Macro: 0.8412
one_vs_one | Poly | Accuracy: 0.7794 | F1 Macro: 0.7790
one_vs_one | Rbf | Accuracy: 0.8968 | F1 Macro: 0.8950
one_vs_one | Sigmoid | Accuracy: 0.5329 | F1 Macro: 0.5294
one_vs_rest | Linear | Accuracy: 0.7974 | F1 Macro: 0.7949
one_vs_rest | Poly | Accuracy: 0.7935 | F1 Macro: 0.7893
one_vs_rest | Rbf | Accuracy: 0.8645 | F1 Macro: 0.8609
one_vs_rest | Sigmoid | Accuracy: 0.4852 | F1 Macro: 0.4877


### Complete Date varying Window Length ###


### Length 1 ###
- Window Length 1
- Complete Features 
- Sample Rate 20Hz and 100Hz
- Sensor Placement Hand and Pocket

In [162]:
WINDOW_LENGTH = 1

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

classifier_train_data = transform_windows_to_features(train_windows)
classifier_test_data = transform_windows_to_features(test_windows)

standard_scaled_train = perform_standard_scaling(classifier_train_data, classifier_train_data)
normalized_train = perform_normalization(standard_scaled_train, standard_scaled_train )

standard_scaled_test = perform_standard_scaling(classifier_train_data, classifier_test_data)
normalized_test = perform_normalization(standard_scaled_train, standard_scaled_test)

In [163]:
x_train_1 = standard_scaled_train.drop(columns=["activity"])
y_train_1 = standard_scaled_train["activity"]

x_test_1 = standard_scaled_test.drop(columns=["activity"])
y_test_1 = standard_scaled_test["activity"]

In [164]:
evaluate_classifiers(x_train_1, y_train_1, x_test_1, y_test_1)

normal | Linear | Accuracy: 0.7620 | F1 Macro: 0.7582
normal | Poly | Accuracy: 0.7441 | F1 Macro: 0.7380
normal | Rbf | Accuracy: 0.8526 | F1 Macro: 0.8501
normal | Sigmoid | Accuracy: 0.5137 | F1 Macro: 0.5155
one_vs_one | Linear | Accuracy: 0.7613 | F1 Macro: 0.7574
one_vs_one | Poly | Accuracy: 0.7913 | F1 Macro: 0.7891
one_vs_one | Rbf | Accuracy: 0.8641 | F1 Macro: 0.8630
one_vs_one | Sigmoid | Accuracy: 0.4997 | F1 Macro: 0.4990
one_vs_rest | Linear | Accuracy: 0.7211 | F1 Macro: 0.7197
one_vs_rest | Poly | Accuracy: 0.7900 | F1 Macro: 0.7854
one_vs_rest | Rbf | Accuracy: 0.8398 | F1 Macro: 0.8373
one_vs_rest | Sigmoid | Accuracy: 0.4276 | F1 Macro: 0.4328


### Length 3 ###
- Window Length 3
- Complete Features 
- Sample Rate 20Hz and 100Hz
- Sensor Placement Hand and Pocket

In [165]:
WINDOW_LENGTH = 3

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

classifier_train_data = transform_windows_to_features(train_windows)
classifier_test_data = transform_windows_to_features(test_windows)

standard_scaled_train = perform_standard_scaling(classifier_train_data, classifier_train_data)
normalized_train = perform_normalization(standard_scaled_train, standard_scaled_train )

standard_scaled_test = perform_standard_scaling(classifier_train_data, classifier_test_data)
normalized_test = perform_normalization(standard_scaled_train, standard_scaled_test)

In [166]:
x_train_3 = standard_scaled_train.drop(columns=["activity"])
y_train_3 = standard_scaled_train["activity"]

x_test_3 = standard_scaled_test.drop(columns=["activity"])
y_test_3 = standard_scaled_test["activity"]

In [167]:
evaluate_classifiers(x_train_3, y_train_3, x_test_3, y_test_3)

normal | Linear | Accuracy: 0.8362 | F1 Macro: 0.8346
normal | Poly | Accuracy: 0.6918 | F1 Macro: 0.6807
normal | Rbf | Accuracy: 0.8556 | F1 Macro: 0.8523
normal | Sigmoid | Accuracy: 0.5754 | F1 Macro: 0.5773
one_vs_one | Linear | Accuracy: 0.8362 | F1 Macro: 0.8345
one_vs_one | Poly | Accuracy: 0.7414 | F1 Macro: 0.7392
one_vs_one | Rbf | Accuracy: 0.8772 | F1 Macro: 0.8760
one_vs_one | Sigmoid | Accuracy: 0.5345 | F1 Macro: 0.5318
one_vs_rest | Linear | Accuracy: 0.7909 | F1 Macro: 0.7878
one_vs_rest | Poly | Accuracy: 0.7780 | F1 Macro: 0.7745
one_vs_rest | Rbf | Accuracy: 0.8491 | F1 Macro: 0.8443
one_vs_rest | Sigmoid | Accuracy: 0.4871 | F1 Macro: 0.4893


###  Length 4 ###
- Window Length 4
- Complete Features 
- Sample Rate 20Hz and 100Hz
- Sensor Placement Hand and Pocket

In [171]:
WINDOW_LENGTH = 4

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

classifier_train_data = transform_windows_to_features(train_windows)
classifier_test_data = transform_windows_to_features(test_windows)

standard_scaled_train = perform_standard_scaling(classifier_train_data, classifier_train_data)
normalized_train = perform_normalization(standard_scaled_train, standard_scaled_train )

standard_scaled_test = perform_standard_scaling(classifier_train_data, classifier_test_data)
normalized_test = perform_normalization(standard_scaled_train, standard_scaled_test)

In [172]:
x_train_4 = standard_scaled_train.drop(columns=["activity"])
y_train_4 = standard_scaled_train["activity"]

x_test_4 = standard_scaled_test.drop(columns=["activity"])
y_test_4 = standard_scaled_test["activity"]

In [173]:
evaluate_classifiers(x_train_4, y_train_4, x_test_4, y_test_4)

normal | Linear | Accuracy: 0.8628 | F1 Macro: 0.8601
normal | Poly | Accuracy: 0.7073 | F1 Macro: 0.6904
normal | Rbf | Accuracy: 0.8293 | F1 Macro: 0.8248
normal | Sigmoid | Accuracy: 0.5762 | F1 Macro: 0.5777
one_vs_one | Linear | Accuracy: 0.8689 | F1 Macro: 0.8666
one_vs_one | Poly | Accuracy: 0.7104 | F1 Macro: 0.7056
one_vs_one | Rbf | Accuracy: 0.8659 | F1 Macro: 0.8639
one_vs_one | Sigmoid | Accuracy: 0.6006 | F1 Macro: 0.5952
one_vs_rest | Linear | Accuracy: 0.8079 | F1 Macro: 0.8062
one_vs_rest | Poly | Accuracy: 0.7744 | F1 Macro: 0.7691
one_vs_rest | Rbf | Accuracy: 0.8384 | F1 Macro: 0.8341
one_vs_rest | Sigmoid | Accuracy: 0.5152 | F1 Macro: 0.5196


### Length 5 ###

In [168]:
WINDOW_LENGTH = 5

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

classifier_train_data = transform_windows_to_features(train_windows)
classifier_test_data = transform_windows_to_features(test_windows)

standard_scaled_train = perform_standard_scaling(classifier_train_data, classifier_train_data)
normalized_train = perform_normalization(standard_scaled_train, standard_scaled_train )

standard_scaled_test = perform_standard_scaling(classifier_train_data, classifier_test_data)
normalized_test = perform_normalization(standard_scaled_train, standard_scaled_test)

In [169]:
x_train_5 = standard_scaled_train.drop(columns=["activity"])
y_train_5 = standard_scaled_train["activity"]

x_test_5 = standard_scaled_test.drop(columns=["activity"])
y_test_5 = standard_scaled_test["activity"]

In [170]:
evaluate_classifiers(x_train_5, y_train_5, x_test_5, y_test_5)

normal | Linear | Accuracy: 0.8381 | F1 Macro: 0.8356
normal | Poly | Accuracy: 0.6763 | F1 Macro: 0.6469
normal | Rbf | Accuracy: 0.8237 | F1 Macro: 0.8192
normal | Sigmoid | Accuracy: 0.5719 | F1 Macro: 0.5725
one_vs_one | Linear | Accuracy: 0.8489 | F1 Macro: 0.8465
one_vs_one | Poly | Accuracy: 0.6906 | F1 Macro: 0.6823
one_vs_one | Rbf | Accuracy: 0.8633 | F1 Macro: 0.8610
one_vs_one | Sigmoid | Accuracy: 0.5612 | F1 Macro: 0.5595
one_vs_rest | Linear | Accuracy: 0.8058 | F1 Macro: 0.8046
one_vs_rest | Poly | Accuracy: 0.7302 | F1 Macro: 0.7275
one_vs_rest | Rbf | Accuracy: 0.8273 | F1 Macro: 0.8219
one_vs_rest | Sigmoid | Accuracy: 0.5000 | F1 Macro: 0.5014


## Sample Rate 100 Hz ##


- Window Length 2
- Complete Features 
- Sample Rate 100Hz
- Sensor Placement Hand and Pocket

In [48]:
WINDOW_LENGTH = 2

train_windows = create_time_windows(train_recordings, WINDOW_LENGTH)
test_windows = create_time_windows(test_recording, WINDOW_LENGTH)

In [49]:
train_windows_100 = [w for w in train_windows if w["sample"].iloc[0] == 100]
test_windows_100 = [w for w in test_windows if w["sample"].iloc[0] == 100]

classifier_train_data_100 = transform_windows_to_features(train_windows_100)
classifier_test_data_100 = transform_windows_to_features(test_windows_100)

standard_scaled_train_100 = perform_standard_scaling(classifier_train_data_100, classifier_train_data_100)
normalized_train_100 = perform_normalization(standard_scaled_train_100, standard_scaled_train_100)

standard_scaled_test_100 = perform_standard_scaling(classifier_test_data_100, classifier_test_data_100)
normalized_test_100 = perform_normalization(standard_scaled_test_100, standard_scaled_test_100)

In [85]:
x_train_100 = standard_scaled_train_100.drop(columns=["activity"])
y_train_100 = standard_scaled_train_100["activity"]

x_test_100 = standard_scaled_test_100.drop(columns=["activity"])
y_test_100 = standard_scaled_test_100["activity"]

In [ ]:
evaluate_classifiers(x_train_100, y_train_100, x_test_100, y_test_100)

## Sample Rate 20 Hz ##


- Window Length 2
- Complete Features 
- Sample Rate 20Hz
- Sensor Placement Hand and Pocket

In [82]:
train_windows_20 = [w for w in train_windows if w["sample"].iloc[0] == 20]
test_windows_20 = [w for w in test_windows if w["sample"].iloc[0] == 20]

classifier_train_data_20 = transform_windows_to_features(train_windows_20)
classifier_test_data_20 = transform_windows_to_features(test_windows_20)

standard_scaled_train_20 = perform_standard_scaling(classifier_train_data_20, classifier_train_data_20)
normalized_train_20 = perform_normalization(standard_scaled_train_20, standard_scaled_train_20)

standard_scaled_test_20 = perform_standard_scaling(classifier_test_data_20, classifier_test_data_20)
normalized_test_20 = perform_normalization(standard_scaled_test_20, standard_scaled_test_20)

In [84]:
x_train_20 = standard_scaled_train_20.drop(columns=["activity"])
y_train_20 = standard_scaled_train_20["activity"]

x_test_20 = standard_scaled_test_20.drop(columns=["activity"])
y_test_20 = standard_scaled_test_20["activity"]

In [ ]:
evaluate_classifiers(x_train_20, y_train_20, x_test_20, y_test_20)

## Compare Results ##

In [136]:
# Window length 1, 2, 3, 4

print("Results for Window =  1")
evaluate_classifiers(x_train_1, y_train_1, x_test_1, y_test_1)
print("Results for Window =  2")
evaluate_classifiers(x_train, y_train, x_test, y_test)
print("Results for Window =  3")
evaluate_classifiers(x_train_3, y_train_3, x_test_3, y_test_3)
print("Results for Window =  4")
evaluate_classifiers(x_train_4, y_train_4, x_test_4, y_test_4)

Results for Window =  1
normal | Linear | Accuracy: 0.7732
normal | Poly | Accuracy: 0.7361
normal | Rbf | Accuracy: 0.8479
normal | Sigmoid | Accuracy: 0.5131
one_vs_one | Linear | Accuracy: 0.7738
one_vs_one | Poly | Accuracy: 0.8051
one_vs_one | Rbf | Accuracy: 0.8875
one_vs_one | Sigmoid | Accuracy: 0.5080
one_vs_rest | Linear | Accuracy: 0.7233
one_vs_rest | Poly | Accuracy: 0.7911
one_vs_rest | Rbf | Accuracy: 0.8422
one_vs_rest | Sigmoid | Accuracy: 0.4275
Results for Window =  2
normal | Linear | Accuracy: 0.8370
normal | Poly | Accuracy: 0.7167
normal | Rbf | Accuracy: 0.8810
normal | Sigmoid | Accuracy: 0.5498
one_vs_one | Linear | Accuracy: 0.8370
one_vs_one | Poly | Accuracy: 0.7633
one_vs_one | Rbf | Accuracy: 0.9133
one_vs_one | Sigmoid | Accuracy: 0.5213
one_vs_rest | Linear | Accuracy: 0.7904
one_vs_rest | Poly | Accuracy: 0.8034
one_vs_rest | Rbf | Accuracy: 0.8693
one_vs_rest | Sigmoid | Accuracy: 0.4580
Results for Window =  3
normal | Linear | Accuracy: 0.8362
norma

## Feature Test ##
Based on Best Result for Window Length 2 and combined 20 and 100 Hz
